In [ ]:
# Uninstall conflicting packages to avoid version mismatches
!pip uninstall -y numpy pandas tensorflow keras

# Install specific versions
!pip install numpy==1.26.4 pandas==2.2.2 tensorflow==2.16.2 opencv-python==4.9.0.80 mediapipe==0.10.14 scikit-learn==1.5.1 tqdm==4.66.5

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.w

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select hand_sign_data.zip
!unzip hand_sign_data.zip

Streaming output truncated to the last 5000 lines.
  inflating: hand_sign_data/thank_you/20/6.npy  
  inflating: hand_sign_data/thank_you/20/7.npy  
  inflating: hand_sign_data/thank_you/20/8.npy  
  inflating: hand_sign_data/thank_you/20/9.npy  
   creating: hand_sign_data/thank_you/21/
  inflating: hand_sign_data/thank_you/21/0.npy  
  inflating: hand_sign_data/thank_you/21/1.npy  
  inflating: hand_sign_data/thank_you/21/10.npy  
  inflating: hand_sign_data/thank_you/21/11.npy  
  inflating: hand_sign_data/thank_you/21/12.npy  
  inflating: hand_sign_data/thank_you/21/13.npy  
  inflating: hand_sign_data/thank_you/21/14.npy  
  inflating: hand_sign_data/thank_you/21/15.npy  
  inflating: hand_sign_data/thank_you/21/16.npy  
  inflating: hand_sign_data/thank_you/21/17.npy  
  inflating: hand_sign_data/thank_you/21/18.npy  
  inflating: hand_sign_data/thank_you/21/19.npy  
  inflating: hand_sign_data/thank_you/21/2.npy  
  inflating: hand_sign_data/thank_you/21/20.npy  
  inflating: h

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models
from tqdm import tqdm
import random

# Configuration
DATA_PATH = '/content/hand_sign_data'
ACTIONS = ['hello', 'please', 'thank_you', 'friend', 'love', 'eat', 'go', 'want', 'happy', 'good', 'what', 'who', 'I', 'you', 'home', 'n']  # 15 ISL signs + 'n'
NUM_SEQUENCES = 30  # Sequences per action
SEQUENCE_LENGTH = 30  # Frames per sequence
LANDMARKS_PER_FRAME = 63  # 21 landmarks × x/y/z
MODEL_PATH = '/content/drive/MyDrive/isl_model.tflite'
BATCH_SIZE = 32
EPOCHS = 50

def load_data():
    """Load landmarks and labels from hand_sign_data"""
    X, y = [], []

    for action in ACTIONS:
        action_path = os.path.join(DATA_PATH, action)
        if not os.path.exists(action_path):
            print(f"Warning: {action_path} not found")
            continue

        for seq in range(NUM_SEQUENCES):
            seq_data = []
            seq_path = os.path.join(action_path, str(seq))
            if not os.path.exists(seq_path):
                print(f"Warning: {seq_path} not found")
                continue

            for frame in range(SEQUENCE_LENGTH):
                frame_path = os.path.join(seq_path, f"{frame}.npy")
                if not os.path.exists(frame_path):
                    print(f"Warning: {frame_path} not found")
                    break
                landmarks = np.load(frame_path)
                if landmarks.shape != (LANDMARKS_PER_FRAME,):
                    print(f"Invalid shape at {frame_path}: {landmarks.shape}")
                    break
                seq_data.append(landmarks)

            if len(seq_data) == SEQUENCE_LENGTH:
                X.append(np.array(seq_data))  # Shape: (30, 63)
                y.append(action)
            else:
                print(f"Skipping incomplete sequence: {seq_path}")

    return np.array(X), np.array(y)

def normalize_landmarks(sequence):
    """Normalize landmarks relative to wrist (landmark 0)"""
    wrist = sequence[:, :3]  # Shape: (30, 3)
    # Tile wrist coordinates to match all 21 landmarks
    wrist_tiled = np.tile(wrist, (1, 21))  # Shape: (30, 63)
    normalized = sequence - wrist_tiled
    scale = np.std(normalized, axis=(0, 1), keepdims=True) + 1e-6
    normalized /= scale
    return normalized

def augment_sequence(sequence):
    """Apply random augmentation: rotation, scaling, noise"""
    augmented = sequence.copy()

    # Random rotation (±10 degrees)
    angle = np.random.uniform(-0.17, 0.17)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    rotation_matrix = np.array([[cos_a, -sin_a, 0], [sin_a, cos_a, 0], [0, 0, 1]])
    for t in range(SEQUENCE_LENGTH):
        for i in range(0, LANDMARKS_PER_FRAME, 3):
            xyz = augmented[t, i:i+3]
            augmented[t, i:i+3] = np.dot(rotation_matrix, xyz)

    # Random scaling (±10%)
    scale = np.random.uniform(0.9, 1.1)
    augmented *= scale

    # Random noise (±0.01)
    noise = np.random.normal(0, 0.01, augmented.shape)
    augmented += noise

    return augmented

def preprocess_data(X, y):
    """Normalize, augment, and encode data"""
    X_normalized = np.array([normalize_landmarks(seq) for seq in X])
    X_averaged = np.mean(X_normalized, axis=1)  # Shape: (samples, 63)

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    y_categorical = tf.keras.utils.to_categorical(y_encoded)

    label_map = dict(zip(ACTIONS, range(len(ACTIONS))))
    np.save('label_map.npy', label_map)

    X_augmented, y_augmented = [], []
    for i in range(len(X_averaged)):
        X_augmented.append(X_averaged[i])
        y_augmented.append(y_categorical[i])
        for _ in range(2):
            X_augmented.append(augment_sequence(X_normalized[i]).mean(axis=0))
            y_augmented.append(y_categorical[i])

    return np.array(X_augmented), np.array(y_augmented), label_encoder

def build_model(num_classes):
    """Build dense model for averaged landmarks"""
    model = models.Sequential([
        layers.Input(shape=(LANDMARKS_PER_FRAME,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def convert_to_tflite(model, X_processed):
    """Convert to TFLite with int8 quantization"""
    def representative_dataset():
        # Yield 100 random samples from X_processed
        indices = np.random.choice(len(X_processed), size=100, replace=False)
        for idx in indices:
            yield [X_processed[idx:idx+1].astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.int8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    converter.representative_dataset = representative_dataset
    tflite_model = converter.convert()
    with open(MODEL_PATH, 'wb') as f:
        f.write(tflite_model)
    print(f"TFLite model saved to {MODEL_PATH}")


In [ ]:
def main():
    print("Loading data...")
    X, y = load_data()
    if len(X) == 0:
        print("Error: No data loaded. Check hand_sign_data folder.")
        return

    print("Preprocessing data...")
    X_processed, y_processed, label_encoder = preprocess_data(X, y)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X_processed, y_processed, test_size=0.2, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    model = build_model(len(ACTIONS))
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

    print("Converting to TFLite...")
    convert_to_tflite(model, X_processed)
    interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    correct = 0
    input_scale, input_zero_point = input_details[0]['quantization']
    output_scale, output_zero_point = output_details[0]['quantization']

    for i in range(len(X_test)):
        # Quantize input to INT8
        input_data = X_test[i:i+1].astype(np.float32)
        input_data = (input_data / input_scale + input_zero_point).astype(np.int8)
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        # Get and dequantize output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale
        pred = np.argmax(output_data)
        true = np.argmax(y_test[i])
        if pred == true:
            correct += 1

    tflite_acc = correct / len(X_test)
    print(f"TFLite test accuracy: {tflite_acc:.4f}")

if __name__ == "__main__":
    main()

Loading data...
Preprocessing data...
Train: 1152, Val: 144, Test: 144
Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.0887 - loss: 3.0282 - val_accuracy: 0.4653 - val_loss: 2.2875
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2556 - loss: 2.3571 - val_accuracy: 0.6250 - val_loss: 1.8983
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2863 - loss: 2.1340 - val_accuracy: 0.7569 - val_loss: 1.4998
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4294 - loss: 1.7600 - val_accuracy: 0.9653 - val_loss: 1.1442
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5356 - loss: 1.4682 - val_accuracy: 0.9583 - val_loss: 0.8515
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5955 - loss: 1.3024 - val_accuracy: 0.9653 - val_loss: 0.6467
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 1.0761 - val_accuracy: 0.9861 - val_loss: 0.5341
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/s

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
files.download('/content/drive/MyDrive/isl_model.tflite')
files.download('/content/drive/MyDrive/label_map.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>